# Прикладной кейс: протеомика стеблей паслёновых + модель GH13

**Цель:** найти опубликованные протеомные данные стеблей растений семейства Solanaceae, извлечь белки-кандидаты GH13, связать их с аминокислотными последовательностями и применить существующий классификатор ESM-2 + Logistic Regression.

## Биологический вопрос
Какие GH13-белки обнаруживаются в стеблях разных паслёновых растений, и какие из них можно приоритизировать для дальнейшего изучения углеводного обмена и потенциальной переработки растительных остатков?

**Важно:** протеомика обычно даёт идентифицированные белки и значения abundance/интенсивности, но не всегда абсолютное содержание в мг/г ткани. Не будем называть относительную интенсивность абсолютным содержанием белка. ML-прогноз — гипотеза, а не экспериментальное подтверждение функции.

## Что будем делать

1. Искать наборы в PRIDE/ProteomeXchange и статьях по `stem`, `shoot`, `Solanaceae`, `tomato`, `potato`, `tobacco`, `pepper`, `eggplant`.
2. Проверять вид, ткань, условия, биологические повторы, количественный метод и доступность processed table.
3. Не смешивать стебли с листьями, корнями, плодами и целыми побегами без явной маркировки.
4. Скачать обработанные таблицы белковых групп/количеств, если они доступны.
5. Сопоставить белковые ID с последовательностями FASTA и аннотациями.
6. Выделить GH13-кандидатов по аннотации и/или профилю семейства.
7. Применить тот же ESM-2 и тот же preprocessing/pooling, что использовались при обучении модели.
8. Объединить прогнозы с abundance и составить список кандидатов для дальнейшего исследования.

## Где искать
- [PRIDE](https://www.ebi.ac.uk/pride/)
- [ProteomeXchange](https://www.proteomexchange.org/)
- [Sol Genomics Network](https://solgenomics.net/)
- [UniProt](https://www.uniprot.org/)

Предварительный веб-поиск находит протеомные наборы плодов томата, например PXD038945 и PXD004932, но это **не стеблевые данные**. Их нельзя включать в основную выборку стеблей; они лишь показывают формат метаданных PRIDE.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("data")
RAW_DIR = PROJECT_DIR / "raw"
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

for path in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Folders ready:", RAW_DIR.resolve(), PROCESSED_DIR.resolve(), RESULTS_DIR.resolve())

Folders ready: /Users/anastasia/work/pet-projects/plant-enzyme-ml/ml/notebooks/data/raw /Users/anastasia/work/pet-projects/plant-enzyme-ml/ml/notebooks/data/processed /Users/anastasia/work/pet-projects/plant-enzyme-ml/ml/notebooks/data/results


## 1. Реестр найденных наборов

Сначала вручную заполняем реестр по страницам PRIDE и публикациям. Автоматический поиск может вернуть плод или лист вместо стебля, поэтому каждый набор надо проверить.

`stem` — явно стебель; `shoot` — побег, состав образца надо уточнить; `other` — другая ткань; `unclear` — пока неясно.

In [2]:
# Добавляйте только реально проверенные записи. Неизвестные значения оставляйте пустыми.
dataset_registry = pd.DataFrame(columns=[
    "project_id", "species", "common_name", "tissue_reported",
    "tissue_match", "condition", "biological_replicates",
    "quantification_type", "processed_table_url", "raw_data_url",
    "publication_doi", "include_primary_analysis", "notes"
])
dataset_registry

,project_id,species,common_name,tissue_reported,tissue_match,condition,biological_replicates,quantification_type,processed_table_url,raw_data_url,publication_doi,include_primary_analysis,notes


### Критерии включения

- Вид относится к Solanaceae.
- Образец — стебель; `shoot` включается только после проверки состава.
- Есть список идентифицированных белков и желательно количественные значения.
- Можно определить биологические повторы и условия.
- Доступна обработанная таблица либо опубликованные дополнительные материалы.

Если общей сопоставимой матрицы для нескольких видов не найдётся, не будем объединять несопоставимые числа. Тогда отдельно анализируем обнаружение белков и количественные данные внутри совместимых экспериментов.

## 2. Загрузить обработанную таблицу

После выбора набора скачайте processed protein groups / protein abundance table в `data/raw/`. Если готовой таблицы нет, проверим, есть ли таблица в supplementary materials.

In [3]:
PROTEOMICS_FILE = None 

if PROTEOMICS_FILE is None:
    print("Файл пока не выбран: сначала проверьте набор и скачайте processed table.")
else:
    file_path = RAW_DIR / PROTEOMICS_FILE
    if not file_path.exists():
        raise FileNotFoundError(file_path.resolve())
    if file_path.suffix.lower() == ".csv":
        proteomics = pd.read_csv(file_path)
    elif file_path.suffix.lower() in {".xlsx", ".xls"}:
        proteomics = pd.read_excel(file_path)
    else:
        proteomics = pd.read_csv(file_path, sep="\t")
    print("Shape:", proteomics.shape)
    display(proteomics.head())
    print("Columns:", proteomics.columns.tolist())

Файл пока не выбран: сначала проверьте набор и скачайте processed table.


## 3. Привести количественные данные к единому формату

Целевой длинный формат: `project_id`, `species`, `tissue`, `sample_id`, `protein_id`, `abundance`, `abundance_type`.

Имена колонок отличаются между публикациями, поэтому mapping заполняется после просмотра конкретного файла. Не сравниваем напрямую LFQ-intensity одного исследования с TMT-значениями другого без обоснованной нормализации.

In [ ]:
# Заполнить после просмотра реальных колонок:
COLUMN_MAP = {
    # "Protein IDs": "protein_id",
    # "Sample A": "sample_A",
    # "Sample B": "sample_B",
}

# Пример после настройки:
# sample_columns = ["sample_A", "sample_B", "sample_C"]
# long_table = proteomics.melt(
#     id_vars=["protein_id"],
#     value_vars=sample_columns,
#     var_name="sample_id",
#     value_name="abundance"
# )

## 4. Получить последовательности и найти GH13

Количественная таблица может содержать только ID, без последовательности. Сопоставляем ID с исходной аннотацией, UniProt/NCBI/SGN и получаем FASTA. Проверяем, что белок относится к нужному виду.

GH13 следует определять по аннотации и/или профилю семейства, например HMMER с подходящим профилем. Одного слова `amylase` в названии недостаточно. Записываем источник каждого ID и последовательности.

In [ ]:
protein_metadata = pd.DataFrame(columns=[
    "protein_id", "species", "gene_name", "protein_name", "sequence",
    "annotation", "gh13_evidence", "source_database", "source_accession"
])
protein_metadata.head()

## 5. Запустить существующую модель

Модель обучена на небольшой выборке из 31 белка и трёх классов, поэтому это прототип. Применять её следует к GH13-кандидатам, а не ко всем белкам стебля.

Критично повторить тот же ESM-2 и тот же способ pooling/preprocessing, что при обучении. Лучше вынести inference в общий модуль, используемый и ноутбуком, и FastAPI.

In [ ]:
# Когда в проекте появится общий inference-модуль:
# from ml.api.inference import predict_sequence
#
# gh13_candidates = protein_metadata[
#     protein_metadata["gh13_evidence"].notna()
#     & protein_metadata["gh13_evidence"].ne("")
# ].copy()
#
# rows = []
# for row in gh13_candidates.itertuples():
#     pred = predict_sequence(row.sequence)
#     rows.append({
#         "protein_id": row.protein_id,
#         "species": row.species,
#         "prediction": pred["prediction"],
#         **{f"score_{k}": v for k, v in pred["probabilities"].items()}
#     })
# prediction_table = pd.DataFrame(rows)
# display(prediction_table.head())

## 6. Объединить прогнозы и abundance

Итоговая таблица должна хранить источник, вид, ткань, условия, повторы, белковый ID, количественное значение, последовательность, свидетельство принадлежности к GH13 и ML-прогноз.

Различаем:
- **обнаружен** — белок идентифицирован в эксперименте;
- **abundance** — измеренное/рассчитанное значение конкретного метода;
- **предсказанный класс** — выход модели;
- **подтверждённая функция** — независимая аннотация или эксперимент.

Не интерпретируем score классификатора как откалиброванную биологическую вероятность.

In [ ]:
# Пример объединения после подготовки таблиц:
# analysis_table = long_table.merge(
#     protein_metadata, on="protein_id", how="left", validate="many_to_one"
# ).merge(
#     prediction_table, on=["protein_id", "species"], how="left",
#     validate="many_to_one"
# )
# analysis_table.to_csv(
#     PROCESSED_DIR / "solanaceae_stem_gh13_candidates.csv", index=False
# )

## 7. Анализ

После загрузки реальных данных:
1. Число обнаруженных GH13-кандидатов по виду.
2. Матрица присутствия/отсутствия белков по образцам.
3. Abundance по биологическим повторам — отдельно для каждого исследования.
4. Список кандидатов с источниками и ML-прогнозами.
5. Межвидовое сравнение только при сопоставимом дизайне и методе.

Не делаем вывод, что вид A содержит больше белка, чем вид B, если измерения получены разными методами, в разных условиях или на разных тканях.

In [ ]:
# Шаблон графика после подготовки analysis_table:
# import matplotlib.pyplot as plt
# candidate = "REAL_PROTEIN_ID"
# plot_df = analysis_table.query("protein_id == @candidate").copy()
# if not plot_df.empty:
#     plot_df.boxplot(column="abundance", by="species", rot=45)
#     plt.title(f"Protein abundance: {candidate}")
#     plt.suptitle("")
#     plt.ylabel("Reported abundance (original units)")
#     plt.tight_layout()
#     plt.show()

## 8. Выходные файлы и критерий успеха

Ожидаемые файлы:
- `dataset_registry.csv` — проверенные источники и метаданные;
- `solanaceae_stem_proteins.csv` — идентифицированные белки и abundance;
- `solanaceae_stem_gh13_candidates.csv` — GH13-кандидаты с последовательностями и ML-прогнозами;
- графики и описание ограничений.

Минимальный успех — воспроизводимая таблица реальных стеблевых протеомных данных хотя бы для нескольких наборов/видов, где каждое измерение связано с источником и условиями. Если сопоставимых стеблевых протеомов мало, это честно указывается. Transcriptomics можно добавить отдельно, но экспрессия РНК не равна содержанию белка.